# Decision Trees & Random Forests

1. **Decision Trees** - Splitting criteria, pruning, visualization
2. **Random Forests** - Bagging, feature importance, OOB error
3. **Comparison** - Single tree vs ensemble

**Dataset**: Wine Quality (multi-class classification)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target
feature_names = wine.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 1. Decision Tree

Trees split data recursively by choosing the feature and threshold that maximize information gain (decrease impurity).

**Gini impurity**: $G = 1 - \sum_{k=1}^{K} p_k^2$  
**Entropy**: $H = -\sum_{k=1}^{K} p_k \log_2(p_k)$

In [ ]:
# Unpruned tree (will overfit)
dt_overfit = DecisionTreeClassifier(random_state=42)
dt_overfit.fit(X_train, y_train)

# Pruned tree
dt_pruned = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
dt_pruned.fit(X_train, y_train)

print(f"Unpruned - Train: {dt_overfit.score(X_train, y_train):.3f}, Test: {dt_overfit.score(X_test, y_test):.3f}")
print(f"Pruned   - Train: {dt_pruned.score(X_train, y_train):.3f}, Test: {dt_pruned.score(X_test, y_test):.3f}")

In [ ]:
# Visualize the pruned tree
plt.figure(figsize=(20, 10))
plot_tree(dt_pruned, feature_names=feature_names, class_names=wine.target_names,
          filled=True, rounded=True, fontsize=9)
plt.title("Pruned Decision Tree")
plt.tight_layout()
plt.show()

## 2. Random Forest

Random Forest = Bagging + Feature Randomness:
- Each tree trained on a **bootstrap sample** (sampling with replacement)
- At each split, only a **random subset of features** is considered
- Final prediction = **majority vote** (classification) or **average** (regression)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=None, min_samples_leaf=2,
    oob_score=True, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

print(f"Train accuracy: {rf.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {rf.score(X_test, y_test):.3f}")
print(f"OOB score:      {rf.oob_score_:.3f}")

print("\n" + classification_report(y_test, rf.predict(X_test), target_names=wine.target_names))

In [ ]:
# Feature importance
importance = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importance.plot(kind="barh", color="teal")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance (mean decrease in impurity)")
plt.tight_layout()
plt.show()

In [ ]:
# Effect of number of trees
n_trees_range = [1, 5, 10, 25, 50, 100, 200, 500]
oob_errors = []

for n in n_trees_range:
    rf_n = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    oob_errors.append(1 - rf_n.oob_score_)

plt.figure(figsize=(8, 5))
plt.plot(n_trees_range, oob_errors, "o-", color="teal")
plt.xlabel("Number of Trees")
plt.ylabel("OOB Error Rate")
plt.title("Random Forest: Error vs. Number of Trees")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Decision trees overfit easily** - always prune (max_depth, min_samples_leaf)
2. **Random forests reduce variance** through averaging many decorrelated trees
3. **OOB score** is a free validation estimate - no need for a separate validation set
4. **Feature importance** from trees is a quick feature selection method
5. **No scaling needed** - trees are invariant to monotonic transformations